### scVI pipeline (single global scVI)
Single scVI model on all cells, batch_key="Study", categorical_covariate_keys=["Tech"].
Matches crc-atlas approach: default params (n_latent=10, n_layers=1), dispersion="gene-batch".

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy as sp
from pathlib import Path
import scvi
import torch
import sys
sys.path.insert(0, "..")
import utils as ut

In [ ]:
# reproducible
scvi.settings.seed = 0
sc.settings.verbosity = 0

In [ ]:
QCDir = Path("../../result/qc/qc.h5ad")
adata = sc.read_h5ad(QCDir)

In [ ]:
# library size normalization + log1p transform
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)
print("Normalization done")

In [ ]:
# ── Per-study HVG intersection ─────────────────────────────────────────
n_hvg_per_study = 4000

hvg_sets = {}
for study in adata.obs["Study"].unique():
    sub = adata[adata.obs["Study"] == study].copy()
    sc.pp.highly_variable_genes(
        sub,
        n_top_genes=n_hvg_per_study,
        flavor="seurat_v3",
        layer="counts",
        subset=False,
    )
    hvg_sets[study] = set(sub.var_names[sub.var["highly_variable"]])
    print(f"{study}: {len(hvg_sets[study])} HVGs")

shared_hvg = sorted(set.intersection(*hvg_sets.values()))
assert len(shared_hvg) > 0, "HVG intersection is empty!"
adata.var["highly_variable"] = adata.var_names.isin(shared_hvg)
print(f"Intersection HVGs: {len(shared_hvg)}")

In [ ]:
# ── Single global scVI ─────────────────────────────────────────────────
# Subset to HVG genes for scVI input
adata_scvi = adata[:, adata.var["highly_variable"]].copy()
print(f"scVI input: {adata_scvi.shape}")

# batch_key="Study" corrects Study effects
# categorical_covariate_keys=["Tech"] explicitly models scRNA/snRNA difference
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key="Study",
    categorical_covariate_keys=["Tech"],
)

# Default params: n_latent=10, n_layers=1 (matching crc-atlas)
# dispersion="gene-batch": gene-specific dispersion, batch-parameterized
model = scvi.model.SCVI(adata_scvi, dispersion="gene-batch")
model.train(accelerator="gpu", max_epochs=400, early_stopping=True)

adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"scVI latent: {adata.obsm['X_scVI'].shape}")

In [ ]:
# ── UMAP + Leiden on scVI latent ──────────────────────────────────────
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden_scVI")
print("UMAP + Leiden done")

In [ ]:
# ── Eval plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
sc.pl.umap(adata, color="Study", ax=axes[0], show=False, title="Study")
sc.pl.umap(adata, color="Tissue", ax=axes[1], show=False, title="Tissue")
sc.pl.umap(adata, color="Tech", ax=axes[2], show=False, title="Tech")
sc.pl.umap(adata, color="leiden_scVI", ax=axes[3], show=False, title="Leiden")
plt.tight_layout()
plt.show()

In [ ]:
# ── Save ───────────────────────────────────────────────────────────────
outdir = Path("../../result/scvi")
outdir.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(outdir / "scvi_adata.h5ad", compression="lzf")
print(f"Saved to {outdir / 'scvi_adata.h5ad'}")
print(f"Cells: {adata.n_obs}, Genes: {adata.n_vars}")
print(f"obsm keys: {list(adata.obsm.keys())}")

In [ ]:
# ── Result check: read back + UMAP ─────────────────────────────────────
adata = sc.read_h5ad("../../result/scvi/scvi_adata.h5ad")

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
sc.pl.umap(adata, color="Study", ax=axes[0], show=False, title="Study")
sc.pl.umap(adata, color="Tissue", ax=axes[1], show=False, title="Tissue")
sc.pl.umap(adata, color="Tech", ax=axes[2], show=False, title="Tech")
sc.pl.umap(adata, color="leiden_scVI", ax=axes[3], show=False, title="Leiden")
plt.tight_layout()
plt.show()